(LaTeX command is defined in this cell.)
$\newcommand{\bm}[1]{\boldsymbol{#1}}$

In [1]:
# --- Environment ---
import sys, time, random
from itertools import combinations

from sage.all import ZZ, QQ, Matrix, vector, binomial
import pandas as pd

# Global log of unauthorized-subset counts, keyed by
#   ('exact', tag, s)  or  ('mc', tag, s, seed)
# where tag identifies which matrix G was used.
UNAUTH_LOG = {}

print("sage   :", sys.version.split()[0], "| pandas:", pd.__version__)
print("UNAUTH_LOG initialised:", UNAUTH_LOG)

sage   : 3.12.11 | pandas: 2.3.2
UNAUTH_LOG initialised: {}


# Initialization

## Basic Helper Function

In [2]:
# --- Basic helpers ---

def linf_norm(v):
    """Infinity norm of a ZZ vector-like object."""
    if len(v) == 0:
        return ZZ(0)
    return max(abs(ZZ(x)) for x in v)

def hamming_weight(v):
    """Number of non-zero entries."""
    return sum(1 for x in v if ZZ(x) != 0)

def sample_unique_subsets(n, t, N, seed=0):
    """N distinct random size-t subsets of {0,...,n-1}, as sorted lists."""
    total = int(binomial(n, t))
    if N > total:
        raise ValueError("N=%d exceeds C(%d,%d)=%d" % (N, n, t, total))
    rng = random.Random(int(seed))
    seen, out = set(), []
    while len(out) < N:
        S = tuple(sorted(rng.sample(range(int(n)), int(t))))
        if S not in seen:
            seen.add(S)
            out.append(list(S))
    return out

Self-check.

In [3]:
# quick self-checks
assert linf_norm(vector(ZZ, [3, -7, 2])) == 7
assert hamming_weight(vector(ZZ, [0, -1, 0, 4])) == 2
assert linf_norm(vector(ZZ, [])) == 0

_s = sample_unique_subsets(10, 4, 5, seed=0)
assert len(_s) == 5 and len(set(map(tuple, _s))) == 5
assert all(len(S) == 4 and S == sorted(S) for S in _s)
print("helpers OK  |  sample:", _s)

helpers OK  |  sample: [[0, 2, 6, 9], [6, 7, 8, 9], [4, 5, 7, 9], [2, 3, 7, 8], [1, 2, 4, 7]]


## HNF Witness

`hnf_witness(G_S)` — authorization test and initial witness.

Computes the row-HNF $\mathbf{R}_{\mathbf{G}_S} = \mathbf{U}_{\mathbf{G}_S}\mathbf{G}_S$ once and uses it for two purposes.

**Authorization.** By the HNF characterization, $\mathbf{e}_1^\top \in \mathrm{Row}_\mathbb{Z}(\mathbf{G}_S)$ if and only if the first row of $\mathbf{R}_{\mathbf{G}_S}$ equals $\mathbf{e}_1^\top$. If it does not, $S$ admits *no* reconstruction vector at all and we return `(None, False)`.

**Witness.** When $S$ is authorized, $\bm{\lambda}_{S,0} := \mathbf{U}_{\mathbf{G}_S}[1,\cdot]$ satisfies $\bm{\lambda}_{S,0}^\top\mathbf{G}_S = \mathbf{e}_1^\top$.

Note that $\mathbf{U}_{\mathbf{G}_S}$ is **not unique**: whenever $|S| > \mathrm{rank}(\mathbf{G}_S)$, adding to row $1$ any row of $\mathbf{U}_{\mathbf{G}_S}$ mapping to a zero row of $\mathbf{R}_{\mathbf{G}_S}$ gives another valid transformation. So $\bm{\lambda}_{S,0}$ is whatever the HNF backend happens to return, and is *not* canonical. Canonicity is restored later by `babai_exact`, which maps every representative of the coset $\bm{\lambda}_{S,0} + \ker_\mathbb{Z}(\mathbf{G}_S^\top)$ to the same vector.

Returning `(None, False)` rather than a vector is the change from the previous notebook, where `hnf_initial_lambda` returned $\mathbf{U}[1,\cdot]$ unconditionally —
so on an unauthorized $S$ it produced a vector satisfying $\bm{\lambda}^\top\mathbf{G}_S = \mathbf{R}_{\mathbf{G}_S}[1,\cdot] \neq \mathbf{e}_1^\top$, which was then counted in the frequency tables.

In [4]:
def hnf_witness(G_S):
    """Row-HNF of G_S computed once.

    Returns (lambda_0, True) with lambda_0^T G_S = e_1^T, or (None, False)
    when e_1^T is not in the integer row span of G_S (S unauthorized).
    """
    if G_S.nrows() == 0:
        return None, False
    R, U = G_S.hermite_form(transformation=True)
    if R.nrows() == 0:
        return None, False
    e1 = vector(ZZ, [1] + [0] * (G_S.ncols() - 1))
    if R.row(0) != e1:
        return None, False
    return vector(ZZ, list(U.row(0))), True

Self-check.

In [5]:
# self-check on a tiny matrix: rows e1, e2, e3, e1+e2, e1+e3, e2+e3
_Gt = Matrix(ZZ, 6, 3, [1,0,0, 0,1,0, 0,0,1, 1,1,0, 1,0,1, 0,1,1])

_l, _ok = hnf_witness(_Gt.matrix_from_rows([0,1,2,3,4,5]))
assert _ok and _l * _Gt.matrix_from_rows([0,1,2,3,4,5]) == vector(ZZ, [1,0,0])

# {r4, r5, r6}: e1 = (r4 + r5 - r6)/2 over QQ, but not over ZZ
_l2, _ok2 = hnf_witness(_Gt.matrix_from_rows([3,4,5]))
assert not _ok2 and _l2 is None

print("hnf_witness OK  |  witness for all 6 rows:", _l)

hnf_witness OK  |  witness for all 6 rows: (1, 0, 0, 0, 0, 0)


## Canonical Basis of Kernel Lattice

`kernel_basis(G_S)` — a canonical basis of the kernel lattice.

Returns a basis of $K_S := \ker_\mathbb{Z}(\mathbf{G}_S^\top) = \{\bm{\nu}\in\mathbb{Z}^{|S|} : \bm{\nu}^\top\mathbf{G}_S = \mathbf{0}^\top\}$ the lattice of vectors that can be added to a reconstruction vector without changing anything. 
If $\bm{\lambda}^\top\mathbf{G}_S = \mathbf{e}_1^\top$ and $\bm{\nu} \in K_S$, then $(\bm{\lambda}+\bm{\nu})^\top\mathbf{G}_S = \mathbf{e}_1^\top$ as well. So the reconstruction vectors for $S$ form the coset $\bm{\lambda}_{S,0} + K_S$, and finding a *short* one means finding a short vector in that coset, an approximate closest-vector problem in $K_S$.

Its rank is $r = |S| - \mathrm{rank}(\mathbf{G}_S)$, which stays small here (6–20 for $n=60$, $k=40$) because $|S|$ only modestly exceeds $k$. That is the structural reason stronger reduction than LLL buys nothing: in dimension under 20 with small entries, an LLL-reduced basis is already close to orthogonal.

**Why `echelonized_basis_matrix()` and not `basis_matrix()`.** A lattice has many bases, and LLL applied to different bases of the same lattice generally returns different reduced bases. The echelonized form is the row-HNF basis, which is *unique* — so it is a canonical function of $\mathbf{G}_S$, and everything computed downstream from it is too. `basis_matrix()` carries no such guarantee.

`use_lll=True` then LLL-reduces it, which improves decoding quality without affecting canonicity: exact LLL at a fixed $\delta$ is a deterministic function of its input basis, and the input is already canonical.

In [6]:
def kernel_basis(G_S, use_lll=True):
    """LLL-reduced row-HNF basis of ker_Z(G_S^T).

    echelonized_basis_matrix() is the row-HNF basis, which is unique --
    that uniqueness is what makes the pipeline canonical.
    """
    B = G_S.transpose().right_kernel(ZZ).echelonized_basis_matrix()
    if use_lll and B.nrows() > 1:
        B = B.LLL()
    return B

Self-check.

In [7]:
# self-check on the tiny matrix from cell 3b
_GS = _Gt.matrix_from_rows([0, 1, 2, 3, 4, 5])
_B  = kernel_basis(_GS)
print("rank r =", _B.nrows(), " (expect |S| - rank =", _GS.nrows() - _GS.rank(), ")")
assert _B.nrows() == _GS.nrows() - _GS.rank()
assert _B * _GS == Matrix(ZZ, _B.nrows(), _GS.ncols())      # every row is in the kernel

# adding a kernel vector preserves the reconstruction equation
_l, _ok = hnf_witness(_GS)
_e1 = vector(ZZ, [1, 0, 0])
assert _l * _GS == _e1
assert (_l + vector(ZZ, list(_B.row(0)))) * _GS == _e1

print("kernel_basis OK")
print(_B.str())

rank r = 3  (expect |S| - rank = 3 )
kernel_basis OK
[ 1  0  1  0 -1  0]
[ 0  1  1  0  0 -1]
[-1 -1  0  1  0  0]


## Babai Exact Algorithm

`babai_exact(B, t)` — canonicalization by nearest-plane decoding.

This is the step that differs most from the earlier approach, in two ways.

**Exact rational arithmetic.** The previous `babai_nearest` began with `t = vector(RealField(128), target)`, truncating the target to about 38 decimal digits. Measurements on the $60\times40$ instance show $\bm{\lambda}_{S,0}$ reaching several hundred bits, so on those subsets the Gram–Schmidt projections $\langle \mathbf{w}, \mathbf{b}^*_i\rangle / \|\mathbf{b}^*_i\|^2$ were computed from a corrupted target. Here everything is in $\mathbb{Q}$, so nothing is lost.

**A fixed tie convention.** Rounding $c$ to $\lfloor c + \tfrac12 \rfloor$ sends half-integers toward $+\infty$, which places the residual in the *half-open* region
\begin{align*}
    \mathcal{F}(\mathbf{B}) = \Bigl\{\textstyle\sum_i x_i\mathbf{b}^*_i \;:\; x_i\in[-\tfrac12,\tfrac12)\Bigr\}
\end{align*}
Half-open matters: $\mathcal{F}(\mathbf{B})$ is then a *fundamental domain* for $L(\mathbf{B})$, meaning each coset of the lattice meets it exactly once.

**Why this gives canonicity.** Nearest-plane returns the unique element of $\mathbf{t} + L(\mathbf{B})$ whose in-span part lies in $\mathcal{F}(\mathbf{B})$, and leaves the component orthogonal to $\mathrm{span}_\mathbb{R}(L(\mathbf{B}))$ untouched. So the output depends only on the *coset* and on $\mathbf{B}$ - not on which representative was supplied. Since two valid transformation matrices differ by $\mathbf{W}$ with $\mathbf{W}\mathbf{G}_S = \mathbf{0}$, their first rows differ by an element of $\ker_\mathbb{Z}(\mathbf{G}_S^\top)$, i.e. lie in the same coset. Feeding either to `babai_exact` therefore gives the same vector, and the non-uniqueness of $\mathbf{U}_{\mathbf{G}_S}$ becomes harmless.

**Quality.** Writing $\mathbf{t} = \mathbf{t}^\perp + \mathbf{t}^\parallel$ for the orthogonal decomposition with respect to $\mathrm{span}_\mathbb{R}(K_S)$, the output
satisfies
\begin{align*}
    \|\mathsf{Babai}(\mathbf{B},\mathbf{t})\|_2^2 \;\le\; \|\mathbf{t}^\perp\|_2^2 + R_\mathrm{Babai}(\mathbf{B})^2, \qquad R_\mathrm{Babai}(\mathbf{B}) = \tfrac12\Bigl(\sum_i\|\mathbf{b}^*_i\|_2^2\Bigr)^{1/2},
\end{align*}
where $\mathbf{t}^\perp$ is the least-norm real solution. So unlike the raw HNF row, the output comes with a proved bound.

The function returns the *residual* (the reduced vector itself), not the lattice offset - the previous `babai_nearest` returned the offset `kappa`, requiring the caller to form `lambda_0 + kappa`.

In [8]:
def babai_exact(B, t):
    """Exact-QQ Babai nearest plane. 
    Returns the residual, an element of t + L(B).
    Ties round toward +infinity, so the in-span part of the output lies in
    {sum x_i b*_i : x_i in [-1/2, 1/2)}, a fundamental domain. Hence the output depends only on the coset t + L(B) and on B.
    """
    r, m = B.nrows(), B.ncols()
    if r == 0:
        return vector(ZZ, list(t))
    b = [vector(QQ, list(B.row(i))) for i in range(r)]
    bs = []
    for i in range(r):                       # Gram-Schmidt over QQ
        v = b[i]
        for j in range(i):
            d = bs[j].dot_product(bs[j])
            if d != 0:
                v = v - (b[i].dot_product(bs[j]) / d) * bs[j]
        bs.append(v)
    w = vector(QQ, list(t))
    for i in reversed(range(r)):             # nearest plane, last index first
        d = bs[i].dot_product(bs[i])
        if d == 0:
            continue
        c  = w.dot_product(bs[i]) / d
        ch = (c + QQ(1) / 2).floor()         # ties toward +infinity
        if ch != 0:
            w = w - QQ(ch) * b[i]
    return vector(ZZ, [ZZ(x) for x in w])

Self-check.

In [9]:
# --- self-check 1: output stays in the coset, and is shorter ---
_l, _ok = hnf_witness(_GS)
_lb = babai_exact(_B, _l)
assert _lb * _GS == _e1
assert linf_norm(_lb) <= linf_norm(_l)

# --- self-check 2: CANONICITY.  Perturbing the witness by kernel vectors
#     reproduces exactly the ambiguity in U; all perturbations must collapse
#     to the same vector. ---
_rnd = random.Random(int(0))
for _ in range(20):
    _pert = vector(ZZ, list(_l))
    for i in range(_B.nrows()):
        _pert += ZZ(int(_rnd.randint(int(-9), int(9)))) * vector(ZZ, list(_B.row(i)))
    assert _pert * _GS == _e1                     # still in the coset
    assert babai_exact(_B, _pert) == _lb          # same canonical output

print("babai_exact OK")
print("  witness  :", _l,  " linf =", linf_norm(_l))
print("  canonical:", _lb, " linf =", linf_norm(_lb))

babai_exact OK
  witness  : (1, 0, 0, 0, 0, 0)  linf = 1
  canonical: (1, 0, 0, 0, 0, 0)  linf = 1


## Greedy-Based Reduction

`greedy_reduce_linf(lam0, B, passes, window)` - local $\ell_\infty$ polish.

Babai's nearest plane minimizes the *Euclidean* norm of the residual, but what enters the signature bound $B_2 \ge n\beta_{\bm{\lambda}}B_1$ is $\|\bm{\lambda}_S\|_\infty$.
The two objectives do not coincide: a vector can be $\ell_2$-short while carrying one large coordinate. This step addresses that directly.

**The move.** Locate the coordinate $j$ where $|\lambda_j|$ is largest. For each kernel basis row $\mathbf{b}_i$ with $b_{i,j}\neq 0$, the choice
\begin{align*}
    \alpha_0 = \Bigl\lfloor \frac{-\lambda_j}{b_{i,j}} + \tfrac12 \Bigr\rfloor
\end{align*}
is the integer multiple that comes closest to zeroing that coordinate. We try $\alpha \in [\alpha_0-\texttt{window},\, \alpha_0+\texttt{window}]$ and keep $\bm{\lambda} + \alpha\mathbf{b}_i$ whenever it strictly lowers $\|\cdot\|_\infty$. Sweeping all rows is one pass; we run up to `passes`, stopping early once a full pass yields no improvement.

**Correctness is automatic.** Every candidate differs from $\bm{\lambda}$ by an integer multiple of a kernel row, so it stays in the coset and continues to satisfy
$\bm{\lambda}^\top\mathbf{G}_S = \mathbf{e}_1^\top$. Only the norm changes.

**Termination.** $\|\bm{\lambda}\|_\infty$ is a non-negative integer that strictly decreases on every accepted move, and `passes` caps the sweeps regardless.

**Canonicity.** The search is deterministic given its inputs, provided the tie-breaks are pinned. Two are needed, and both are already in place: `max(range(...), key=...)` returns the *first* maximizer, so ties on $j$ go to the smallest index; and $\alpha$ is enumerated in increasing order with *strict* improvement required, so the earliest $\alpha$ wins a tie. The $\alpha_0$ rounding uses `(QQ(...) + 1/2).floor()` rather than Python's `round`, which applies banker's rounding to a `Rational` and would not match the convention used in
`babai_exact`.

**Note on `passes=2, window=2`.** These match the values used in the earlier experiments, so the results are comparable. The polish is cheap relative to the HNF, so raising them costs little if a harder instance warrants it.

In [10]:
def greedy_reduce_linf(lam0, B, passes=2, window=2):
    """Local search: add integer multiples of kernel rows to shrink ||.||_inf.

    Stays in the coset by construction, so the reconstruction equation is preserved.  
    Tie-breaks: argmax to the smallest index, alpha ascending with strict improvement.
    """
    lam  = vector(ZZ, list(lam0))
    best = linf_norm(lam)
    if B.nrows() == 0:
        return lam
    for _ in range(int(passes)):
        improved = False
        for i in range(B.nrows()):
            bi = vector(ZZ, list(B.row(i)))
            j  = max(range(len(lam)), key=lambda t: abs(lam[t]))
            if bi[j] == 0:
                continue
            a0 = (QQ(-lam[j]) / QQ(bi[j]) + QQ(1) / 2).floor()
            lam_best, norm_best = lam, best
            for a in range(int(a0 - window), int(a0 + window) + 1):
                cand = lam + ZZ(a) * bi
                cn   = linf_norm(cand)
                if cn < norm_best:
                    lam_best, norm_best = cand, cn
            if norm_best < best:
                lam, best, improved = lam_best, norm_best, True
        if not improved:
            break
    return lam

Self-check.

In [11]:
# --- self-check: coset preserved, norm never increases, deterministic ---
_lg = greedy_reduce_linf(_lb, _B, passes=2, window=2)
assert _lg * _GS == _e1
assert linf_norm(_lg) <= linf_norm(_lb)
assert greedy_reduce_linf(_lb, _B, passes=2, window=2) == _lg   # same input, same output

# starting from a deliberately inflated representative, it should come back down
_bad = _lb + ZZ(37) * vector(ZZ, list(_B.row(0)))
assert _bad * _GS == _e1
_fix = greedy_reduce_linf(_bad, _B, passes=4, window=4)
assert _fix * _GS == _e1

print("greedy_reduce_linf OK")
print("  babai     :", _lb,  " linf =", linf_norm(_lb))
print("  + greedy  :", _lg,  " linf =", linf_norm(_lg))
print("  inflated  :", _bad, " linf =", linf_norm(_bad))
print("  recovered :", _fix, " linf =", linf_norm(_fix))

greedy_reduce_linf OK
  babai     : (1, 0, 0, 0, 0, 0)  linf = 1
  + greedy  : (1, 0, 0, 0, 0, 0)  linf = 1
  inflated  : (38, 0, 37, 0, -37, 0)  linf = 38
  recovered : (0, 0, -1, 0, 1, 0)  linf = 1


## Full Pipeline: Construction of $\bm{\lambda}_S$

`construct_lambda(G, S, passes, window)` — the full pipeline

Chains the four previous functions:

1. `hnf_witness` — one row-HNF. Returns `None` immediately if $S$ is unauthorized.
2. `kernel_basis` — canonical (row-HNF) basis of $K_S$, then LLL-reduced.
3. `babai_exact` — canonicalizes the witness and produces a short representative.
4. `greedy_reduce_linf` — local $\ell_\infty$ polish.

The output is a function of $(\mathbf{G}, S)$ and the fixed parameters alone. Each ingredient is either unique as a mathematical object (the two row-HNFs) or a deterministic function of a canonical input (exact LLL at fixed $\delta$, Babai with a pinned tie convention, greedy with pinned tie-breaks). Nothing here depends on a library version or a floating-point precision setting, which is what lets the paper specify the algorithm rather than pin an implementation.

**Return value.** A vector $\bm{\lambda}_S \in \mathbb{Z}^{|S|}$ with $\bm{\lambda}_S^\top\mathbf{G}_S = \mathbf{e}_1^\top$ exactly, or `None` when $\mathbf{e}_1^\top \notin \mathrm{Row}_\mathbb{Z}(\mathbf{G}_S)$.

Callers **must** handle the `None` case. Counting it as a coefficient would mix two different populations into one histogram: reconstruction vectors for authorized
$S$, and 0 in the earlier code, which returned $\mathbf{U}[1,\cdot]$ regardless - vectors satisfying $\bm{\lambda}^\top\mathbf{G}_S = \mathbf{R}_{\mathbf{G}_S}[1,\cdot]
\neq \mathbf{e}_1^\top$, which are not reconstruction vectors at all.

**What was measured on the $60\times40$ instance.** Over 366,051 subsets at sizes
46–60: $\beta_{\bm{\lambda}} = 8$ and $\omega_{\bm{\lambda}} = 42$, both attained at
$s=46$; the unauthorized fraction falls from $0.63\%$ at $s=46$ to zero by $s=52$;
and $\Pr[\|\bm{\lambda}_S\|_\infty \le 2] = 1$ for every $s \ge 50$. The present
notebook repeats this for the other matrices.

In [12]:
def construct_lambda(G, S, passes=2, window=2):
    """Canonical reconstruction vector for G_S, or None if S is unauthorized.

    HNF witness -> canonical LLL'd kernel basis -> exact Babai -> greedy polish.
    The output depends only on (G, S, passes, window).
    """
    S = list(S)
    assert len(S) > 0
    G_S = G.matrix_from_rows(S)
    lam0, ok = hnf_witness(G_S)
    if not ok:
        return None
    B = kernel_basis(G_S, use_lll=True)
    if B.nrows() == 0:
        return lam0
    return greedy_reduce_linf(babai_exact(B, lam0), B, passes=passes, window=window)

Self-check.

In [13]:
# --- self-checks on the toy matrix ---
_lam = construct_lambda(_Gt, [0, 1, 2, 3, 4, 5])
assert _lam is not None and _lam * _Gt.matrix_from_rows([0,1,2,3,4,5]) == _e1

assert construct_lambda(_Gt, [3, 4, 5]) is None          # unauthorized
assert construct_lambda(_Gt, [0, 1, 2]) is not None      # r = 0, trivial kernel

# determinism: same inputs, same output
assert construct_lambda(_Gt, [0,1,2,3,4,5]) == _lam

print("construct_lambda OK")
print("  all 6 rows :", _lam, " linf =", linf_norm(_lam),
      " wt =", hamming_weight(_lam))
print("  {r4,r5,r6} :", construct_lambda(_Gt, [3, 4, 5]), "(unauthorized)")
print("  {r1,r2,r3} :", construct_lambda(_Gt, [0, 1, 2]), "(r = 0)")

construct_lambda OK
  all 6 rows : (1, 0, 0, 0, 0, 0)  linf = 1  wt = 1
  {r4,r5,r6} : None (unauthorized)
  {r1,r2,r3} : (1, 0, 0) (r = 0)


# The Matrix

We consider the parity check matrix of an $(n,m,c,\gamma,(1-\epsilon)c)$ lossless bipartite expander with:
1. $n = 50$
2. $m = 24$
3. $c = 3$
4. $\gamma = 5/50 = 0.1000$
5. $\epsilon = 0.470$

The matrix $\mathbf{G} = \begin{bmatrix}\mathbf{g} \ \mathbf{\hat{G}}\end{bmatrix} \in \mathbb{Z}^{n \times k}$ where $\mathbf{\hat{G}} = (\mathbf{P}\mathbf{H})^\top$ for some signed permutation matrix $\mathbf{P}$. 

In [14]:
rows = [
    [ 1,-1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],  # r1
    [ 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0,-1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0],  # r2
    [ 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,-1,-1, 0],  # r3
    [ 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,-1, 0, 0, 0, 1, 0, 0, 0, 0],  # r4
    [ 1, 0, 0, 0, 0, 0, 0, 0,-1, 0, 0, 0, 0, 0,-1, 0, 0, 0, 0, 0, 0, 0,-1, 0, 0],  # r5
    [ 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0,-1, 0, 0, 0, 1, 0, 0, 0, 0],  # r6
    [ 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0,-1, 0, 0,-1, 0, 0, 0, 0, 0],  # r7
    [ 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0,-1],  # r8
    [ 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0,-1, 0],  # r9
    [ 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,-1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0,-1, 0],  # r10
    [ 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0,-1, 0],  # r11
    [-1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0,-1, 0, 0, 0, 0,-1, 0, 0, 0, 0, 0],  # r12
    [ 1, 0, 0, 0, 0, 0, 0, 0,-1, 1, 0, 0, 0, 0, 0, 0,-1, 0, 0, 0, 0, 0, 0, 0, 0],  # r13
    [ 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0,-1, 0, 0, 0, 0,-1],  # r14
    [ 1,-1, 0, 0, 0, 0, 0, 0,-1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],  # r15
    [ 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0],  # r16
    [ 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0,-1, 0, 0],  # r17
    [ 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0,-1, 0,-1, 0, 0, 0, 0, 0, 0, 0, 0],  # r18
    [-1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0,-1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],  # r19
    [ 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1,-1, 1, 0, 0, 0, 0],  # r20
    [ 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0,-1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0],  # r21
    [ 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,-1, 0, 0, 0],  # r22
    [ 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,-1, 0, 0, 0, 0, 0],  # r23
    [ 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0,-1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,-1],  # r24
    [ 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0],  # r25
    [ 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,-1, 0,-1, 0],  # r26
    [-1,-1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0,-1, 0],  # r27
    [ 1,-1, 0, 0,-1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],  # r28
    [ 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,-1, 0, 0,-1, 1, 0, 0, 0, 0, 0, 0, 0],  # r29
    [ 1,-1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,-1, 0, 0],  # r30
    [ 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0,-1, 0, 0, 0, 0, 0, 0, 0, 0],  # r31
    [-1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,-1, 0],  # r32
    [ 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,-1, 0],  # r33
    [ 1, 0, 0, 1,-1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,-1, 0, 0, 0, 0, 0],  # r34
    [ 1, 0, 0, 0, 0, 0, 0, 0,-1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,-1],  # r35
    [-1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,-1, 0, 0, 0, 0, 0, 0, 0, 0, 0,-1,-1, 0],  # r36
    [ 1, 0, 0, 0,-1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],  # r37
    [ 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0,-1, 0],  # r38
    [ 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0,-1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],  # r39
    [ 1, 0, 1, 0,-1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],  # r40
    [ 0, 0, 0, 0, 0, 0, 0, 0,-1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0,-1, 0, 0],  # r41
    [ 0, 0, 0, 1, 0, 0, 0, 0,-1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],  # r42
    [ 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1,-1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],  # r43
    [ 0, 0, 0, 0,-1, 0, 0, 0, 0, 0, 0, 0,-1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],  # r44
    [ 0, 0, 1, 0, 0, 0, 0, 0,-1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,-1],  # r45
    [ 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0,-1],  # r46
    [ 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0,-1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0],  # r47
    [ 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,-1, 0, 1, 0, 0, 0,-1, 0, 0, 0, 0, 0],  # r48
    [ 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0,-1, 0, 0, 0, 0, 0, 0, 0,-1, 0, 0, 0],  # r49
    [ 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0,-1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],  # r50
]

G = Matrix(ZZ, rows)

TAG = "G50x25"          # identifies this matrix in UNAUTH_LOG and in table captions

print(TAG, ":", G.nrows(), "x", G.ncols(), "over", G.base_ring())
print("rank:", G.rank())
print("entries in:", sorted(set(ZZ(x) for x in G.list())))
print("max |entry|:", max(abs(ZZ(x)) for x in G.list()))

G50x25 : 50 x 25 over Integer Ring
rank: 25
entries in: [-1, 0, 1]
max |entry|: 1


# Frequency Tables

Frequency tables for one subset size.

For a fixed $s$, we either enumerate all $\binom{n}{s}$ subsets or sample $N$ distinct ones, compute $\bm{\lambda}_S$ for each authorized $S$, and accumulate two histograms:
- `linf_freq[v]` — how many $S$ gave $\|\bm{\lambda}_S\|_\infty = v$,
- `wt_freq[w]` — how many $S$ gave $\mathrm{wt}_H(\bm{\lambda}_S) = w$.

Each table has an overflow bin at its last index counting everything beyond the range, so a value outside the expected range is visible rather than lost. From the histograms,
\begin{align*}
    \beta_{\bm{\lambda}}(s) = \max\{v : \texttt{linf\_freq}[v] > 0\},\qquad
  \omega_{\bm{\lambda}}(s) = \max\{w : \texttt{wt\_freq}[w] > 0\},
\end{align*}
and the reported $\beta_{\bm{\lambda}}, \omega_{\bm{\lambda}}$ are the maxima over all $s$ tested.

**Unauthorized subsets are skipped**, not counted. The tables therefore describe the distribution *conditional on* $S$ being authorized, and $\beta_{\bm{\lambda}}$ is a
maximum over $\Gamma_1$ — matching the definition. The count of skipped subsets is recorded separately in `UNAUTH_LOG`; it is the quantity that determines $\epsilon_c$, and keeping it apart from the coefficient distribution means both numbers mean what the definitions say. (In the earlier $60\times40$ run, failing to separate them inflated two entries of the published table: single unauthorized subsets appeared as $\|\bm{\lambda}_S\|_\infty = 3$ at $s=50$ and $s=51$, where the true maximum over authorized subsets is $2$.)

As a consistency check the function asserts `sum(linf_freq) == sum(wt_freq)`, which catches any double-counting.

In [15]:
def freq_table_for_s(G, s, tag, N=None, seed=0, passes=2, window=2,
                     linf_max_bin=15, wt_max_bin=None, verbose=True):
    """Histograms of ||lambda_S||_inf and wt_H(lambda_S) over subsets of size s.

    N=None  -> exhaustive over all C(n,s) subsets
    N=int   -> N distinct sampled subsets (falls back to exhaustive if N >= C(n,s))

    Returns dict with: s, mode, tested, authorized, unauthorized,
                       linf_max, omega, linf_freq, wt_freq.
    Also writes (unauth, auth) into UNAUTH_LOG under (mode, tag, s, seed).
    """
    n   = G.nrows()
    s   = int(s)
    tot = int(binomial(n, s))
    if wt_max_bin is None:
        wt_max_bin = s

    if N is None or N >= tot:
        subs, mode = combinations(range(n), s), "exact"
        tested_target = tot
    else:
        subs, mode = sample_unique_subsets(n, s, int(N), seed), "mc"
        tested_target = int(N)

    L, W = int(linf_max_bin), int(wt_max_bin)
    linf_freq = [0] * (L + 2)          # 0..L, overflow at L+1
    wt_freq   = [0] * (W + 2)          # 0..W, overflow at W+1
    n_auth = n_unauth = 0

    t0 = time.time()
    for S in subs:
        lam = construct_lambda(G, S, passes=passes, window=window)
        if lam is None:
            n_unauth += 1
            continue
        n_auth += 1
        a = int(linf_norm(lam))
        w = int(hamming_weight(lam))
        linf_freq[a if a <= L else L + 1] += 1
        wt_freq[w if w <= W else W + 1] += 1
    dt = time.time() - t0

    assert sum(linf_freq) == sum(wt_freq) == n_auth
    assert n_auth + n_unauth == tested_target

    linf_max = max((i for i, c in enumerate(linf_freq) if c > 0), default=None)
    omega    = max((i for i, c in enumerate(wt_freq)   if c > 0), default=None)

    UNAUTH_LOG[(mode, tag, s, int(seed))] = (n_unauth, n_auth)
    if verbose:
        print("s=%2d [%5s] tested=%7d auth=%7d unauth=%5d | linf_max=%-3s omega=%-3s | %6.1fs"
              % (s, mode, tested_target, n_auth, n_unauth, linf_max, omega, dt))

    return dict(s=s, mode=mode, tested=tested_target, authorized=n_auth,
                unauthorized=n_unauth, linf_max=linf_max, omega=omega,
                linf_freq=linf_freq, wt_freq=wt_freq, secs=dt)

Self-check.

In [16]:
# --- self-check: exhaustive on the toy matrix ---
_r = freq_table_for_s(_Gt, 5, "toy", N=None, linf_max_bin=5, wt_max_bin=5)
assert _r['authorized'] + _r['unauthorized'] == int(binomial(6, 5)) == 6
assert sum(_r['linf_freq']) == _r['authorized']
print("freq_table_for_s OK |", {k: _r[k] for k in
      ('mode','tested','authorized','unauthorized','linf_max','omega')})

s= 5 [exact] tested=      6 auth=      6 unauth=    0 | linf_max=1   omega=2   |    0.0s
freq_table_for_s OK | {'mode': 'exact', 'tested': 6, 'authorized': 6, 'unauthorized': 0, 'linf_max': 1, 'omega': 2}


# Sweeping Range of Sizes

Sweeping a range of sizes

`freq_curve` calls `freq_table_for_s` for each $s$ in a range and collects the results.

**Repeats.** In Monte Carlo mode each repeat draws $N$ distinct subsets under the seed $\texttt{seed} + 1000s + r$; the histograms are then summed across repeats.
Within one repeat the $N$ subsets are distinct, but different repeats may overlap, so the union is not guaranteed to be exactly $N \cdot \texttt{repeats}$ distinct subsets. How much this matters depends on $\binom{n}{s}$: for $n=60$, $N=10000$, `repeats=3` the expected duplicate count was under 100 for $s \le 55$ and about 900 at $s=56$ where $\binom{60}{56}$ had fallen to $4.9\times10^5$. For $n=40$ the binomials are smaller throughout, so check $\binom{40}{s}$ against $N\cdot r$ before assuming the samples are effectively distinct — and prefer exhaustive mode once $\binom{n}{s}$ is within reach.

**Exhaustive mode** ignores `repeats`, since enumerating the same set three times would just triple the runtime.

**Bin sizes.** `linf_max_bin` should exceed the largest $\|\bm{\lambda}_S\|_\infty$ you expect ($8$ on the $60\times40$ instance, so $15$ was ample) and `wt_max_bin` the largest weight ($42$ there, so $40$ was *not* ample — two subsets landed in the overflow bin and the exact value could not be read off). Leave headroom: an overflow count tells you a value was exceeded but not by how much.

In [17]:
def freq_curve(G, tag, s_range, N=None, seed=0, repeats=1, passes=2, window=2, linf_max_bin=15, wt_max_bin=None,verbose=True):
    """Run freq_table_for_s over a range of sizes, merging repeats.

    N=None      -> exhaustive at every size (repeats ignored)
    N=int       -> Monte Carlo, `repeats` samples per size, histograms summed
    wt_max_bin  -> None means use s at each size (exact table, no overflow)

    Returns a list of per-size dicts, same keys as freq_table_for_s.
    """
    out = []
    for s in s_range:
        tot = int(binomial(G.nrows(), s))
        n_rep = 1 if (N is None or N >= tot) else int(repeats)

        merged = None
        for r in range(n_rep):
            res = freq_table_for_s(
                G, s, tag, N=N, seed=int(seed) + 1000 * int(s) + r,
                passes=passes, window=window,
                linf_max_bin=linf_max_bin,
                wt_max_bin=wt_max_bin,
                verbose=(verbose and n_rep == 1))
            if merged is None:
                merged = res
            else:
                merged['linf_freq'] = [a + b for a, b in
                                       zip(merged['linf_freq'], res['linf_freq'])]
                merged['wt_freq']   = [a + b for a, b in
                                       zip(merged['wt_freq'], res['wt_freq'])]
                for k in ('tested', 'authorized', 'unauthorized', 'secs'):
                    merged[k] += res[k]

        merged['linf_max'] = max((i for i, c in enumerate(merged['linf_freq']) if c > 0),
                                 default=None)
        merged['omega']    = max((i for i, c in enumerate(merged['wt_freq']) if c > 0),
                                 default=None)
        merged['repeats']  = n_rep
        assert sum(merged['linf_freq']) == sum(merged['wt_freq']) == merged['authorized']

        if verbose and n_rep > 1:
            print("s=%2d [%5s x%d] tested=%7d auth=%7d unauth=%5d | "
                  "linf_max=%-3s omega=%-3s | %6.1fs"
                  % (s, merged['mode'], n_rep, merged['tested'], merged['authorized'],
                     merged['unauthorized'], merged['linf_max'], merged['omega'],
                     merged['secs']))
        out.append(merged)
    return out

Self-check.

In [18]:
# --- self-check: exhaustive over the toy matrix, sizes 4..6 ---
_c = freq_curve(_Gt, "toy", range(4, 7), N=None, linf_max_bin=5, wt_max_bin=6)
assert [d['s'] for d in _c] == [4, 5, 6]
assert [d['tested'] for d in _c] == [int(binomial(6, s)) for s in (4, 5, 6)]
print("freq_curve OK")

# --- self-check: MC mode falls back to exhaustive when N >= C(n,s) ---
_c2 = freq_curve(_Gt, "toy", [5], N=10**6, repeats=3, linf_max_bin=5, wt_max_bin=6)
assert _c2[0]['mode'] == "exact" and _c2[0]['repeats'] == 1
print("MC fallback OK")

s= 4 [exact] tested=     15 auth=     15 unauth=    0 | linf_max=1   omega=2   |    0.0s
s= 5 [exact] tested=      6 auth=      6 unauth=    0 | linf_max=1   omega=2   |    0.0s
s= 6 [exact] tested=      1 auth=      1 unauth=    0 | linf_max=1   omega=1   |    0.0s
freq_curve OK
s= 5 [exact] tested=      6 auth=      6 unauth=    0 | linf_max=1   omega=2   |    0.0s
MC fallback OK


# Summary Table

Summary table

`curve_to_frame` turns the list of per-size dicts into a `DataFrame` with one row
per $s$:

| column | meaning |
|---|---|
| `s` | subset size |
| `mode` | `exact` or `mc` (with `repeats`) |
| `tested` | subsets drawn or enumerated |
| `auth` / `unauth` | authorized / unauthorized counts |
| `unauth_%` | percentage unauthorized — the quantity behind $\epsilon_c$ |
| `beta_s` | $\max_S \|\bm{\lambda}_S\|_\infty$ at this size |
| `omega_s` | $\max_S \mathrm{wt}_H(\bm{\lambda}_S)$ at this size |
| `P[linf=1]` | fraction with unit coefficients |
| `P[linf<=2]` | cumulative fraction at 2 |
| `mean_linf` / `mean_wt` | means over authorized subsets |
| `secs` | wall time |

The headline parameters are the column maxima:
$\beta_{\bm{\lambda}} = \max_s \texttt{beta\_s}$ and
$\omega_{\bm{\lambda}} = \max_s \texttt{omega\_s}$.

`overflow_linf` and `overflow_wt` report the last-bin counts. **Both must be zero**; a non-zero entry means a value exceeded the histogram range and `beta_s` or `omega_s` is a floor rather than the true maximum. Re-run that size with a larger bin if so.

In [19]:
def curve_to_frame(curve, tag=None):
    """Per-size summary DataFrame from a freq_curve result."""
    recs = []
    for d in curve:
        lf, wf = d['linf_freq'], d['wt_freq']
        a = d['authorized']
        recs.append({
            's':            d['s'],
            'mode':         d['mode'] + (" x%d" % d['repeats'] if d.get('repeats', 1) > 1 else ""),
            'tested':       d['tested'],
            'auth':         a,
            'unauth':       d['unauthorized'],
            'unauth_%':     round(100.0 * d['unauthorized'] / d['tested'], 4) if d['tested'] else 0.0,
            'beta_s':       d['linf_max'],
            'omega_s':      d['omega'],
            'P[linf=1]':    round(lf[1] / a, 4) if a else None,
            'P[linf<=2]':   round(sum(lf[:3]) / a, 4) if a else None,
            'mean_linf':    round(sum(i * c for i, c in enumerate(lf)) / a, 3) if a else None,
            'mean_wt':      round(sum(i * c for i, c in enumerate(wf)) / a, 2) if a else None,
            'overflow_linf': lf[-1],
            'overflow_wt':   wf[-1],
            'secs':         round(d['secs'], 1),
        })
    df = pd.DataFrame(recs).set_index('s')
    if tag is not None:
        df.attrs['tag'] = tag
    return df

def report(curve, tag=""):
    """Print the summary table and the headline parameters."""
    df = curve_to_frame(curve, tag)
    display(df)
    beta  = max(d['linf_max'] for d in curve)
    omega = max(d['omega']    for d in curve)
    tot_t = sum(d['tested'] for d in curve)
    tot_u = sum(d['unauthorized'] for d in curve)
    ov    = df['overflow_linf'].sum() + df['overflow_wt'].sum()
    print("%s: beta_lambda = %d   omega_lambda = %d" % (tag, beta, omega))
    print("   over %d subsets, %d unauthorized (%.4f%%)"
          % (tot_t, tot_u, 100.0 * tot_u / tot_t))
    if ov:
        print("   WARNING: %d subsets fell in an overflow bin -- widen the bins "
              "and re-run; beta/omega are lower bounds." % ov)
    else:
        print("   overflow bins empty: beta and omega are exact.")
    return df

def freq_frame(curve, which='linf', normalize=False):
    """Full histogram as a DataFrame: rows = s, columns = bin value."""
    key = 'linf_freq' if which == 'linf' else 'wt_freq'
    rows = {}
    for d in curve:
        v = d[key]
        rows[d['s']] = ([c / d['authorized'] for c in v] if normalize and d['authorized']
                        else list(v))
    df = pd.DataFrame(rows).T
    df.index.name = 's'
    df.columns.name = ('||lambda||_inf' if which == 'linf' else 'wt_H')
    return df.loc[:, (df != 0).any(axis=0)]        # drop all-zero columns

Self-check.

In [20]:
# --- self-check on the toy curve from cell 10b ---
_df = report(_c, "toy 6x3")
assert list(_df.index) == [4, 5, 6]
print()
display(freq_frame(_c, 'linf'))

,mode,tested,auth,unauth,unauth_%,beta_s,omega_s,P[linf=1],P[linf<=2],mean_linf,mean_wt,overflow_linf,overflow_wt,secs
s,,,,,,,,,,,,,,
4,exact,15,15,0,0.0,1,2,1.0,1.0,1.0,1.33,0,0,0.0
5,exact,6,6,0,0.0,1,2,1.0,1.0,1.0,1.17,0,0,0.0
6,exact,1,1,0,0.0,1,1,1.0,1.0,1.0,1.0,0,0,0.0


toy 6x3: beta_lambda = 1   omega_lambda = 2
   over 22 subsets, 0 unauthorized (0.0000%)
   overflow bins empty: beta and omega are exact.



||lambda||_inf,1
s,
4,15
5,6
6,1


# Testing for $n = 50$ Data

## Sanity Check for Small Data

$s \in [31,39]$ using MC.

In [21]:
%%time
curve_p1 = freq_curve(G, TAG, range(31, 39), N=100, seed=0, repeats=3,
                      passes=2, window=2, linf_max_bin=15, wt_max_bin=None)

s=31 [   mc x3] tested=    300 auth=    297 unauth=    3 | linf_max=2   omega=26  |    0.7s
s=32 [   mc x3] tested=    300 auth=    298 unauth=    2 | linf_max=2   omega=22  |    0.8s
s=33 [   mc x3] tested=    300 auth=    299 unauth=    1 | linf_max=2   omega=20  |    1.0s
s=34 [   mc x3] tested=    300 auth=    300 unauth=    0 | linf_max=2   omega=22  |    1.1s
s=35 [   mc x3] tested=    300 auth=    300 unauth=    0 | linf_max=2   omega=20  |    1.2s
s=36 [   mc x3] tested=    300 auth=    300 unauth=    0 | linf_max=2   omega=18  |    1.4s
s=37 [   mc x3] tested=    300 auth=    300 unauth=    0 | linf_max=1   omega=16  |    1.6s
s=38 [   mc x3] tested=    300 auth=    300 unauth=    0 | linf_max=1   omega=20  |    1.9s
CPU times: user 9.78 s, sys: 21.5 ms, total: 9.8 s
Wall time: 9.8 s


$s \in [48,50]$ full exhaustive search.

In [22]:
%%time
curve_p2 = freq_curve(G, TAG, range(48, 51), N=None,
                      passes=2, window=2, linf_max_bin=15, wt_max_bin=None)

s=48 [exact] tested=   1225 auth=   1225 unauth=    0 | linf_max=1   omega=14  |   17.9s
s=49 [exact] tested=     50 auth=     50 unauth=    0 | linf_max=1   omega=12  |    0.8s
s=50 [exact] tested=      1 auth=      1 unauth=    0 | linf_max=1   omega=6   |    0.0s
CPU times: user 18.7 s, sys: 19.5 ms, total: 18.7 s
Wall time: 18.7 s


## Full Run

$s \in [31,39]$ using MC.

In [23]:
%%time
curve_p1 = freq_curve(G, TAG, range(31, 40), N=10000, seed=0, repeats=3,
                      passes=2, window=2, linf_max_bin=15, wt_max_bin=None)

s=31 [   mc x3] tested=  30000 auth=  29831 unauth=  169 | linf_max=4   omega=27  |   83.1s
s=32 [   mc x3] tested=  30000 auth=  29928 unauth=   72 | linf_max=3   omega=26  |   91.4s
s=33 [   mc x3] tested=  30000 auth=  29965 unauth=   35 | linf_max=4   omega=28  |  104.2s
s=34 [   mc x3] tested=  30000 auth=  29982 unauth=   18 | linf_max=2   omega=26  |  117.8s
s=35 [   mc x3] tested=  30000 auth=  29995 unauth=    5 | linf_max=2   omega=26  |  131.5s
s=36 [   mc x3] tested=  30000 auth=  29997 unauth=    3 | linf_max=2   omega=26  |  147.6s
s=37 [   mc x3] tested=  30000 auth=  29998 unauth=    2 | linf_max=2   omega=24  |  168.6s
s=38 [   mc x3] tested=  30000 auth=  30000 unauth=    0 | linf_max=2   omega=24  |  188.1s
s=39 [   mc x3] tested=  30000 auth=  30000 unauth=    0 | linf_max=2   omega=22  |  205.2s
CPU times: user 20min 37s, sys: 2.04 s, total: 20min 39s
Wall time: 20min 39s


$s \in [40,46]$ using MC.

In [24]:
%%time
curve_p2 = freq_curve(G, TAG, range(40, 47), N=10000, seed=0, repeats=3,
                      passes=2, window=2, linf_max_bin=15, wt_max_bin=None)

s=40 [   mc x3] tested=  30000 auth=  30000 unauth=    0 | linf_max=2   omega=24  |  220.2s
s=41 [   mc x3] tested=  30000 auth=  30000 unauth=    0 | linf_max=2   omega=22  |  244.1s
s=42 [   mc x3] tested=  30000 auth=  30000 unauth=    0 | linf_max=1   omega=22  |  269.3s
s=43 [   mc x3] tested=  30000 auth=  30000 unauth=    0 | linf_max=1   omega=18  |  291.9s
s=44 [   mc x3] tested=  30000 auth=  30000 unauth=    0 | linf_max=1   omega=22  |  315.6s
s=45 [   mc x3] tested=  30000 auth=  30000 unauth=    0 | linf_max=1   omega=20  |  348.0s
s=46 [   mc x3] tested=  30000 auth=  30000 unauth=    0 | linf_max=1   omega=18  |  385.7s
CPU times: user 34min 34s, sys: 2.13 s, total: 34min 36s
Wall time: 34min 36s


$s \in [47,50]$ using full exhaustive search.

In [25]:
%%time
curve_p3 = freq_curve(G, TAG, range(47, 51), N=None,
                      passes=2, window=2, linf_max_bin=15, wt_max_bin=None)

s=47 [exact] tested=  19600 auth=  19600 unauth=    0 | linf_max=1   omega=18  |  277.2s
s=48 [exact] tested=   1225 auth=   1225 unauth=    0 | linf_max=1   omega=14  |   19.0s
s=49 [exact] tested=     50 auth=     50 unauth=    0 | linf_max=1   omega=12  |    0.8s
s=50 [exact] tested=      1 auth=      1 unauth=    0 | linf_max=1   omega=6   |    0.0s
CPU times: user 4min 56s, sys: 384 ms, total: 4min 57s
Wall time: 4min 57s


In [26]:
df = report(curve_p1 + curve_p2 + curve_p3, TAG)

,mode,tested,auth,unauth,unauth_%,beta_s,omega_s,P[linf=1],P[linf<=2],mean_linf,mean_wt,overflow_linf,overflow_wt,secs
s,,,,,,,,,,,,,,
31,mc x3,30000,29831,169,0.5633,4,27,0.7892,0.9982,1.213,12.0,0,0,83.1
32,mc x3,30000,29928,72,0.24,3,26,0.896,0.9997,1.104,11.26,0,0,91.4
33,mc x3,30000,29965,35,0.1167,4,28,0.9584,0.9999,1.042,10.62,0,0,104.2
34,mc x3,30000,29982,18,0.06,2,26,0.9843,1.0,1.016,10.07,0,0,117.8
35,mc x3,30000,29995,5,0.0167,2,26,0.994,1.0,1.006,9.55,0,0,131.5
36,mc x3,30000,29997,3,0.01,2,26,0.9978,1.0,1.002,9.15,0,0,147.6
37,mc x3,30000,29998,2,0.0067,2,24,0.9995,1.0,1.001,8.78,0,0,168.6
38,mc x3,30000,30000,0,0.0,2,24,0.9997,1.0,1.0,8.48,0,0,188.1
39,mc x3,30000,30000,0,0.0,2,22,0.9999,1.0,1.0,8.24,0,0,205.2


G50x25: beta_lambda = 4   omega_lambda = 28
   over 500876 subsets, 304 unauthorized (0.0607%)
   overflow bins empty: beta and omega are exact.
